# QC results 

Summarizes QC rating across raters for a given dataset. Estimates inter-rate-reliability and other metrics based on available subject overlaps. 

Gathers every rater's QC output listed in `RATER_CONFIG` and builds dataset-level reports:

* **Which file is used.** Files live in each rater's `out_dir` (default `<rater_dir>/qc-output`) and are grouped per
  QC task by name: `<rater_id>_<pipeline>_<task>_qc_status.tsv` and `checkpoints/<rater_id>_<pipeline>_<task>_checkpoint_<stamp>.tsv`.
  For each rater and task, the `qc_status.tsv` is used when it exists; otherwise the newest checkpoint (by the stamp
  in its filename). With `PREFER_TRANSFORMED_CHECKPOINTS`, a checkpoint's `*_transformed.tsv` copy (current facet
  names) replaces the original of the same stamp.
* **Sanity check.** `qc_status.tsv` is rewritten on every save, so it should hold everything the newest checkpoint
  holds; `in_latest_ckpt_not_in_status` counts the participants for which it doesn't.
* **Comparisons.** Rated participants are checked against each rater's own `participant_list` and against the
  dataset-level `PARTICIPANT_LIST`. Comparison is on `participant_id` because the lists don't carry sessions.
* Repeated rows for the same rater, participant, session and facet keep the latest `timestamp`.

In [ ]:
# --- Local paths --------------------------------------------------------------
# Machine-specific paths live in notebooks/local_config.json (gitignored).
# Create it by copying notebooks/local_config.example.json and editing the paths.
import json
from pathlib import Path

LOCAL_CONFIG = next(
    (p for p in (Path("local_config.json"), Path("notebooks") / "local_config.json") if p.is_file()), None
)
if LOCAL_CONFIG is None:
    raise FileNotFoundError(
        "notebooks/local_config.json not found: copy notebooks/local_config.example.json and set your paths"
    )
_local = json.loads(LOCAL_CONFIG.read_text())
print(f"Local config: {LOCAL_CONFIG.resolve()}")

RATER_CONFIG = _local["rater_config"]            # raters whose outputs are collated
PARTICIPANT_LIST = _local["participant_list"]    # dataset-level participant pool
OUTPUT_DIR = _local.get("output_dir")            # write report TSVs here (None = don't save)

# --- Settings ---------------------------------------------------------------
# When the chosen checkpoint has a *_transformed.tsv copy (current facet names), use it
PREFER_TRANSFORMED_CHECKPOINTS = False


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reuse the app's code so results match QC-Studio and its admin dashboard exactly.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ui" / "utils" / "user_config.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "ui"))
from utils.assignments import read_participant_ids
from utils.ratings_report import (
    QC_ORDER,
    VALUE_ORDER,
    agreement_summary,
    assigned_ids,
    assignment_completion,
    collate_ratings,
    consensus,
    dataset_coverage,
    facet_rating_counts,
    final_qc_counts,
    off_list_ratings,
    participant_level,
    rater_progress,
    coverage_buckets,
)
from utils.user_config import load_users_config

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

# Status colours (good / warning / serious / critical) and a neutral grey for missing.
QC_COLORS = {"All-Pass": "#0ca30c", "Mixed": "#fab219", "All-Uncertain": "#ec835a", "All-Fail": "#d03b3b", "All-Missing": "#898781"}
VALUE_COLORS = {"PASS": "#0ca30c", "UNCERTAIN": "#ec835a", "FAIL": "#d03b3b", "MISSING": "#898781"}

## 1. Find and load each rater's current rating file

In [ ]:
cfg = load_users_config(RATER_CONFIG)
pool_ids = read_participant_ids(PARTICIPANT_LIST)
assigned = assigned_ids(cfg)
print(f"Dataset participant list: {PARTICIPANT_LIST}  ({len(pool_ids)} participants)")

col = collate_ratings(cfg, prefer_transformed=PREFER_TRANSFORMED_CHECKPOINTS)
ratings, sources_df = col.ratings, col.sources
for w in col.warnings:
    print(f"WARNING {w}")
print(f"Raters with rating files: {sources_df['rater_id'].nunique()} / {len(cfg.users)}")
if col.raters_without_files:
    print(f"No status/checkpoint files yet: {', '.join(col.raters_without_files)}")
print(f"Facet-level ratings: {len(ratings)} rows ({col.n_duplicates_dropped} duplicate rows dropped)")
display(sources_df)
flag = sources_df[sources_df["in_latest_ckpt_not_in_status"] > 0]
if not flag.empty:
    print("WARNING: newest checkpoint has participants missing from the status file:")
    display(flag)

## 2. Participant-level ratings

One row per rater × task × participant/session, with its `final_qc` and the number of failing, uncertain and missing facets.

In [ ]:
qc = participant_level(ratings)
tasks = sorted(qc["qc_task"].unique())
print(f"Participant-level ratings: {len(qc)} (rater x task x participant/session)")
display(final_qc_counts(qc))

## 3. Rater progress

Per rater and task: participants rated out of the rater's own list, plus anyone rated outside that list or outside the dataset list.

In [ ]:
print("Completion: rated / assigned participants, summed over raters (raters who haven't started count as 0)")
display(assignment_completion(qc, assigned))

progress = rater_progress(qc, assigned, pool_ids)
display(progress)
off = off_list_ratings(qc, assigned, pool_ids)
if not off.empty:
    print("Raters who rated participants outside their own list or outside the dataset list:")
    display(off)

# Heatmap: % of each rater's list completed per task (blank = task not started)
all_raters = sorted(u["rater_id"] for u in cfg.users.values())
pct = progress.pivot(index="rater_id", columns="qc_task", values="pct_complete").reindex(index=all_raters, columns=tasks)
cnt = progress.pivot(index="rater_id", columns="qc_task", values="rated_assigned").reindex_like(pct)
fig, ax = plt.subplots(figsize=(1.6 * len(tasks) + 3, 0.35 * len(pct) + 1.5))
im = ax.imshow(pct.values.astype(float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(tasks)), tasks)
ax.set_yticks(range(len(pct)), pct.index)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]):
        v = pct.iat[i, j]
        ax.text(j, i, "–" if pd.isna(v) else f"{v:.0f}% ({int(cnt.iat[i, j])})", ha="center", va="center",
                fontsize=8, color="white" if (not pd.isna(v) and v > 60) else "#52514e")
fig.colorbar(im, ax=ax, label="% of assigned list rated")
ax.set_title("Rater progress by QC task: % of own list rated (n rated)")
plt.tight_layout()
plt.show()

## 4. Dataset coverage against `PARTICIPANT_LIST`

How many participants in the dataset list have been rated per task, and by how many raters.

In [ ]:
coverage_qc = dataset_coverage(qc, pool_ids, assigned)
display(coverage_qc)
assigned_not_in_pool = set().union(*assigned.values()) - set(pool_ids)
if assigned_not_in_pool:
    print(f"WARNING: {len(assigned_not_in_pool)} participants in rater lists are not in the dataset list, "
          f"e.g. {sorted(assigned_not_in_pool)[:5]}")

# Stacked bars: how many raters each pool participant has per task
dist = coverage_buckets(qc, pool_ids, tasks)
fig, ax = plt.subplots(figsize=(8, 0.6 * len(tasks) + 1.5))
left = np.zeros(len(dist))
for b, color in zip(dist.columns, ["#e1e0d9", "#9ec5f4", "#3987e5", "#184f95"]):
    ax.barh(dist.index, dist[b], left=left, color=color, edgecolor="white", linewidth=2, label=f"{b} raters")
    left += dist[b].values
ax.set_xlabel(f"Participants in dataset list (n={len(set(pool_ids))})")
ax.set_title("Dataset coverage: raters per participant")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.25), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Rating distributions

`final_qc` mix per rater (to spot lenient or strict raters), and the facet-level rating mix per task (which facets fail most).

In [ ]:
def stacked_share(ax, table, order, colors, title):
    share = table.reindex(columns=order, fill_value=0)
    share = 100 * share.div(share.sum(axis=1), axis=0)
    left = np.zeros(len(share))
    for c in order:
        ax.barh(share.index, share[c], left=left, color=colors[c], edgecolor="white", linewidth=2, label=c)
        left += share[c].values
    ax.set_xlim(0, 100)
    ax.set_title(title, fontsize=10)
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)

# final_qc mix per rater, one panel per task (spots lenient / strict raters)
fig, axes = plt.subplots(1, len(tasks), figsize=(5 * len(tasks), 0.35 * qc["rater_id"].nunique() + 2), squeeze=False)
for ax, t in zip(axes[0], tasks):
    g = qc[qc["qc_task"] == t]
    tab = pd.crosstab(g["rater_id"], g["final_qc"])
    stacked_share(ax, tab, QC_ORDER, QC_COLORS, t)
    ax.set_yticks(range(len(tab)), [f"{r} (n={n})" for r, n in tab.sum(axis=1).items()])
    ax.set_xlabel("% of rated participants")
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(QC_ORDER), loc="lower center", frameon=False)
fig.suptitle("final_qc by rater")
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

# Facet-level rating mix per task (least-passing facets on top)
facet_tabs = {t: facet_rating_counts(ratings, t) for t in tasks}
fig, axes = plt.subplots(len(tasks), 1, figsize=(9, sum(0.28 * len(f) + 1 for f in facet_tabs.values())),
                         squeeze=False, gridspec_kw={"height_ratios": [max(len(f), 1) for f in facet_tabs.values()]})
for ax, (t, tab) in zip(axes[:, 0], facet_tabs.items()):
    n = qc[qc["qc_task"] == t].shape[0]
    stacked_share(ax, tab, VALUE_ORDER, VALUE_COLORS, f"{t} (n={n} ratings)")
    ax.tick_params(axis="y", labelsize=8)
axes[-1, 0].set_xlabel("% of facet ratings (least-passing facets on top)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(VALUE_ORDER), loc="lower center", frameon=False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 6. Participants rated by more than one rater

`majority_qc` is the `final_qc` given by more than half of the raters, else `No majority`.

In [ ]:
cons = consensus(qc)
multi = cons[cons["n_raters"] > 1]
print(f"Participants rated by >1 rater: {len(multi)} (task x participant/session)")
display(agreement_summary(cons))
display(multi[~multi["unanimous"]].head(20))

## 7. Save reports (when `OUTPUT_DIR` is set)

In [ ]:
if OUTPUT_DIR:
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    for name, df in {
        "qc_ratings_facets": ratings, "qc_ratings_participants": qc, "qc_consensus": cons,
        "qc_rater_progress": progress, "qc_dataset_coverage": coverage_qc, "qc_source_files": sources_df,
    }.items():
        df.to_csv(out / f"{name}.tsv", sep="\t", index=False)
        print(f"wrote {out / f'{name}.tsv'} ({len(df)} rows)")
else:
    print("OUTPUT_DIR is None: reports not saved")